In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
# 전처리 완료 파일 경로
preprocessed_path = '/content/drive/MyDrive/itda/yelpzip_preprocessed.csv'

df = pd.read_csv(preprocessed_path)

df.head()

,user_id,prod_id,rating,label,date,text,review_id
0,5044,0,1.0,1,2014-11-16,"Drinks were bad, the hot chocolate was watered...",0
1,5045,0,1.0,1,2014-09-08,This was the worst experience I've ever had a ...,1
2,5046,0,3.0,1,2013-10-06,This is located on the site of the old Spruce ...,2
3,5047,0,5.0,1,2014-11-30,I enjoyed coffee and breakfast twice at Toast ...,3
4,5048,0,5.0,1,2014-08-28,I love Toast! The food choices are fantastic -...,4


In [5]:
df['date'] = pd.to_datetime(df['date'])

print(df.info())
print(df['label'].value_counts())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 608458 entries, 0 to 608457
Data columns (total 7 columns):
 #   Column     Non-Null Count   Dtype         
---  ------     --------------   -----         
 0   user_id    608458 non-null  int64         
 1   prod_id    608458 non-null  int64         
 2   rating     608458 non-null  float64       
 3   label      608458 non-null  int64         
 4   date       608458 non-null  datetime64[ns]
 5   text       608458 non-null  object        
 6   review_id  608458 non-null  int64         
dtypes: datetime64[ns](1), float64(1), int64(4), object(1)
memory usage: 32.5+ MB
None
label
0    528019
1     80439
Name: count, dtype: int64


샘플링

In [7]:
#샘플 사이즈는 15000개로 함
SAMPLE_SIZE = 15000

# product별 리뷰 수 계산
prod_counts = df['prod_id'].value_counts()

print("product 개수:", len(prod_counts))
print(prod_counts.head(10))

# 리뷰 수가 많은 product 순서
top_products = prod_counts.index.tolist()

sampled_parts = []
current_count = 0

for prod in top_products:
    prod_df = df[df['prod_id'] == prod].copy()

    remaining = SAMPLE_SIZE - current_count

    if remaining <= 0:
        break

    # 이 product의 리뷰를 전부 넣어도 15000개를 넘지 않으면 전부 추가
    if len(prod_df) <= remaining:
        sampled_parts.append(prod_df)
        current_count += len(prod_df)

    # 넘으면 필요한 개수만 날짜순으로 추가
    else:
        prod_df = prod_df.sort_values('date')
        sampled_parts.append(prod_df.iloc[:remaining])
        current_count += remaining
        break

sampled_df = pd.concat(sampled_parts, axis=0)

print("샘플링 후 크기:", sampled_df.shape)

product 개수: 5044
prod_id
3745    7378
4698    6632
3237    4716
3136    3935
1881    3143
3875    3122
2605    2999
3876    2959
828     2943
4223    2848
Name: count, dtype: int64
샘플링 후 크기: (15000, 7)


In [8]:
# 기존 review_id는 추적용으로 저장해놓음
if 'review_id' in sampled_df.columns:
    sampled_df = sampled_df.rename(columns={'review_id': 'original_review_id'})

# 날짜순/상품순 정렬 후 index 초기화
sampled_df = sampled_df.sort_values(['prod_id', 'date']).reset_index(drop=True)

# GNN node 번호용 review_id 생성
sampled_df['review_id'] = sampled_df.index

print(sampled_df.head())
print(sampled_df[['review_id', 'user_id', 'prod_id', 'rating', 'label', 'date']].head())

   user_id  prod_id  rating  label       date  \
0     8253     3237     5.0      0 2005-08-13   
1    17480     3237     3.0      0 2005-09-19   
2   149466     3237     5.0      0 2005-10-11   
3    41746     3237     5.0      0 2005-10-27   
4   192388     3237     5.0      1 2005-10-28   

                                                text  original_review_id  \
0  One of the original pizza places. Worth the lo...              378701   
1  Expect a wait before your first bite into this...              378700   
2  Could this pizza be anymore delicious?  The wh...              378699   
3  Shocker!  Everyone thinks that this is best pi...              378698   
4  One of my favorite places in the city.  The pi...              374686   

   review_id  
0          0  
1          1  
2          2  
3          3  
4          4  
   review_id  user_id  prod_id  rating  label       date
0          0     8253     3237     5.0      0 2005-08-13
1          1    17480     3237     3.0      

In [9]:
print("샘플링 데이터 크기:", sampled_df.shape)

print("\nlabel 분포:")
print(sampled_df['label'].value_counts())
print(sampled_df['label'].value_counts(normalize=True))

print("\nuser 수:", sampled_df['user_id'].nunique())
print("product 수:", sampled_df['prod_id'].nunique())

print("\n상위 product 리뷰 수:")
print(sampled_df['prod_id'].value_counts().head(10))

샘플링 데이터 크기: (15000, 8)

label 분포:
label
0    13432
1     1568
Name: count, dtype: int64
label
0    0.895467
1    0.104533
Name: proportion, dtype: float64

user 수: 13620
product 수: 3

상위 product 리뷰 수:
prod_id
3745    7378
4698    6632
3237     990
Name: count, dtype: int64


train / valid / test split

In [10]:


from sklearn.model_selection import train_test_split

RANDOM_STATE = 42

# 전체 index
indices = sampled_df.index.values
labels = sampled_df['label'].values

# 먼저 train_valid 80%, test 20%
train_valid_idx, test_idx = train_test_split(
    indices,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=labels
)

# train_valid 안에서 train 80%, valid 20%
# 전체 기준으로 train 64%, valid 16%
train_valid_labels = sampled_df.loc[train_valid_idx, 'label'].values

train_idx, valid_idx = train_test_split(
    train_valid_idx,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=train_valid_labels
)

print("train 개수:", len(train_idx))
print("valid 개수:", len(valid_idx))
print("test 개수:", len(test_idx))
print("총합:", len(train_idx) + len(valid_idx) + len(test_idx))

train 개수: 9600
valid 개수: 2400
test 개수: 3000
총합: 15000


In [11]:
#split 컬럼과 mask 컬럼 생성
sampled_df['split'] = 'none'

sampled_df.loc[train_idx, 'split'] = 'train'
sampled_df.loc[valid_idx, 'split'] = 'valid'
sampled_df.loc[test_idx, 'split'] = 'test'

sampled_df['train_mask'] = sampled_df['split'] == 'train'
sampled_df['valid_mask'] = sampled_df['split'] == 'valid'
sampled_df['test_mask'] = sampled_df['split'] == 'test'

print(sampled_df['split'].value_counts())

split
train    9600
test     3000
valid    2400
Name: count, dtype: int64


In [12]:
#split별 label 분포 확인
print("전체 label 분포")
print(sampled_df['label'].value_counts(normalize=True))

print("\ntrain label 분포")
print(sampled_df[sampled_df['split'] == 'train']['label'].value_counts(normalize=True))

print("\nvalid label 분포")
print(sampled_df[sampled_df['split'] == 'valid']['label'].value_counts(normalize=True))

print("\ntest label 분포")
print(sampled_df[sampled_df['split'] == 'test']['label'].value_counts(normalize=True))

전체 label 분포
label
0    0.895467
1    0.104533
Name: proportion, dtype: float64

train label 분포
label
0    0.895521
1    0.104479
Name: proportion, dtype: float64

valid label 분포
label
0    0.895417
1    0.104583
Name: proportion, dtype: float64

test label 분포
label
0    0.895333
1    0.104667
Name: proportion, dtype: float64


저장

In [13]:
save_path = '/content/drive/MyDrive/itda/itda_sampling_split.csv'

sampled_df.to_csv(save_path, index=False, encoding='utf-8-sig')